In [1]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.metrics import mean_absolute_error

In [2]:

class MLOpsMonitor:

    def __init__(
        self, baseline_csv_path="data/processed/cleaned_traffic_features.csv"
    ):
        # Load reference baseline training data
        try:
            self.baseline_df = pd.read_csv(baseline_csv_path)
        except FileNotFoundError:
            # Generate synthetic baseline if file missing
            np.random.seed(42)
            self.baseline_df = pd.DataFrame(
                {
                    "temp_scaled": np.random.normal(0.5, 0.1, 1000),
                    "traffic_volume": np.random.randint(1000, 5000, 1000),
                }
            )

        self.baseline_mae = 198.35  # Recorded production benchmark MAE
        self.ks_threshold_pvalue = 0.05  # Standard alpha for statistical drift
        self.mae_degradation_tolerance = 0.20  # 20% error increase threshold

    def simulate_production_batch(self, introduce_drift=False):
        """Simulate a new production batch of incoming inference data."""
        batch = self.baseline_df.copy().sample(n=300, random_state=101)

        if introduce_drift:
            # Simulate sudden sensor shift in temperature & unexpected congestion spike
            batch["temp_scaled"] = batch["temp_scaled"] + np.random.normal(
                0.35, 0.05, len(batch)
            )
            batch["actual_volume"] = batch[
                "traffic_volume"
            ] * 1.35 + np.random.normal(0, 300, len(batch))
            batch["predicted_volume"] = batch["traffic_volume"]
        else:
            # Normal distribution with minor random noise
            batch["actual_volume"] = batch[
                "traffic_volume"
            ] + np.random.normal(0, 150, len(batch))
            batch["predicted_volume"] = batch["traffic_volume"]

        return batch

    def evaluate_system_health(self, production_batch):
        """Perform statistical drift tests and error evaluation."""
        alerts = []
        status = "PASS / Normal"

        # 1. Feature Distribution Drift Test (KS-Test on Key Features)
        ks_stat, p_value = ks_2samp(
            self.baseline_df["temp_scaled"], production_batch["temp_scaled"]
        )
        feature_drift_detected = p_value < self.ks_threshold_pvalue

        if feature_drift_detected:
            status = "ALERT / Requires Investigation"
            alerts.append(
                f"FEATURE DRIFT DETECTED in 'temp_scaled' (KS Stat: {ks_stat:.4f}, p-value: {p_value:.4e})"
            )

        # 2. Prediction Error Drift Test (MAE Tracking)
        current_mae = mean_absolute_error(
            production_batch["actual_volume"],
            production_batch["predicted_volume"],
        )
        mae_increase_pct = (
            current_mae - self.baseline_mae
        ) / self.baseline_mae

        if mae_increase_pct > self.mae_degradation_tolerance:
            status = "ALERT / Requires Investigation"
            alerts.append(
                f"PREDICTION ERROR DRIFT DETECTED: MAE degraded by {mae_increase_pct*100:.1f}% "
                f"(Baseline MAE: {self.baseline_mae:.2f} -> Current MAE: {current_mae:.2f})"
            )

        return {
            "system_status": status,
            "feature_drift_p_value": round(p_value, 5),
            "baseline_mae": self.baseline_mae,
            "current_mae": round(current_mae, 2),
            "mae_degradation_pct": f"{mae_increase_pct*100:.1f}%",
            "active_alerts": (
                alerts
                if alerts
                else ["None - All metrics within operational thresholds."]
            ),
        }


# Execute Monitoring Simulation
monitor = MLOpsMonitor()

print("=" * 70)
print("SCENARIO 1: NORMAL PRODUCTION BATCH EVALUATION")
print("=" * 70)
batch_normal = monitor.simulate_production_batch(introduce_drift=False)
report_normal = monitor.evaluate_system_health(batch_normal)
print(f"System Operational Status : {report_normal['system_status']}")
print(f"Feature KS-Test p-value   : {report_normal['feature_drift_p_value']}")
print(
    f"Model MAE (Baseline/Curr) : {report_normal['baseline_mae']} / {report_normal['current_mae']}"
)
print(f"Alert Log                 : {report_normal['active_alerts'][0]}\n")

print("=" * 70)
print("SCENARIO 2: DRIFTED PRODUCTION BATCH EVALUATION")
print("=" * 70)
batch_drifted = monitor.simulate_production_batch(introduce_drift=True)
report_drifted = monitor.evaluate_system_health(batch_drifted)
print(f"System Operational Status : {report_drifted['system_status']}")
print(f"Feature KS-Test p-value   : {report_drifted['feature_drift_p_value']}")
print(
    f"Model MAE (Baseline/Curr) : {report_drifted['baseline_mae']} / {report_drifted['current_mae']}"
)
print("Alert Log Messages:")
for alert in report_drifted["active_alerts"]:
    print(f"  - 🔴 {alert}")

SCENARIO 1: NORMAL PRODUCTION BATCH EVALUATION
System Operational Status : PASS / Normal
Feature KS-Test p-value   : 0.08973
Model MAE (Baseline/Curr) : 198.35 / 114.27
Alert Log                 : None - All metrics within operational thresholds.

SCENARIO 2: DRIFTED PRODUCTION BATCH EVALUATION
System Operational Status : ALERT / Requires Investigation
Feature KS-Test p-value   : 0.00011
Model MAE (Baseline/Curr) : 198.35 / 1163.33
Alert Log Messages:
  - 🔴 FEATURE DRIFT DETECTED in 'temp_scaled' (KS Stat: 0.1277, p-value: 1.0805e-04)
  - 🔴 PREDICTION ERROR DRIFT DETECTED: MAE degraded by 486.5% (Baseline MAE: 198.35 -> Current MAE: 1163.33)
